In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import random
from collections import deque
import tkinter as tk
import time
import requests
import base64
import signal

In [3]:
API_KEY = 'W8JCC52R'
BASE_URL = 'http://141.211.79.101:10002/v1'
USERNAME = 'user4'
PASSWORD = 'user4'


AUTHORIZATION = {
    'Authorization': 'Basic ' + base64.b64encode(f"{USERNAME}:{PASSWORD}".encode()).decode()
}

class RITGateway:
    def __init__(self, username, password, host='http://141.211.79.101:10002', use_http=True):
        protocol = 'http' if use_http else 'https'
        self.base_url = f'{protocol}://{host.split("://")[-1]}/v1'
        self.session = requests.Session()

        auth_value = base64.b64encode(f'{username}:{password}'.encode()).decode()
        self.session.headers.update({
            'Authorization': f'Basic {auth_value}'
        })

    def _request(self, method, endpoint, params=None):
        url = f'{self.base_url}/{endpoint}'

        while True:
            if method == 'GET':
                resp = self.session.get(url, params=params)
            elif method == 'POST':
                resp = self.session.post(url, params=params)
            elif method == 'DELETE':
                resp = self.session.delete(url, params=params)
            else:
                raise ValueError(f'Unsupported HTTP method: {method}')

            if resp.status_code == 401:
                raise ApiException('Authentication failed. Check username/password.')

            if resp.status_code == 429:
                wait_time = float(resp.headers.get('Retry-After', 1))
                print(f'Rate limit hit. Sleeping {wait_time} sec.')
                time.sleep(wait_time)
                continue

            if not resp.ok:
                raise ApiException(f'API request failed: {resp.status_code} {resp.text}')

            if resp.text.strip():
                return resp.json()
            return None

    def get_case(self):
        return self._request('GET', 'case')

    def get_securities(self):
        return self._request('GET', 'securities')

    def get_security(self, ticker):
        securities = self.get_securities()
        for sec in securities:
            if sec.get('ticker') == ticker:
                return sec
        return None

    def get_book(self, ticker):
        return self._request('GET', 'securities/book', params={'ticker': ticker})

    def get_orders(self, status=None):
        params = {}
        if status is not None:
            params['status'] = status
        return self._request('GET', 'orders', params=params)

    def submit_order(self, ticker, order_type, quantity, action, price=None):
        params = {
            'ticker': ticker,
            'type': order_type,
            'quantity': int(quantity),
            'action': action
        }
        if price is not None:
            params['price'] = round(float(price), 2)

        return self._request('POST', 'orders', params=params)

    def cancel_all_orders(self):
        return self._request('POST', 'commands/cancel', params={'all': 1})

    def cancel_order(self, order_id):
        return self._request('DELETE', f'orders/{order_id}')

In [4]:
class Quoter:
    """
    LT2 version:
    - Only posts passive unwind orders on the side that reduces inventory.
    - Uses small slices because GTA is illiquid.
    - Can scale aggressiveness based on urgency.
    """

    def __init__(
        self,
        base_post_size=2000,
        max_post_size=8000,
        inventory_limit=50000,
        min_clip=500,
        price_improve=0.01
    ):
        self.base_post_size = base_post_size
        self.max_post_size = max_post_size
        self.inventory_limit = inventory_limit
        self.min_clip = min_clip
        self.price_improve = price_improve

    def _round_lot(self, qty):
        return max(0, int(qty / 100) * 100)

    def _compute_post_size(self, inventory, urgency):
        """
        urgency in [0,1]
        """
        abs_inv = abs(inventory)
        inv_ratio = min(1.0, abs_inv / self.inventory_limit)

        raw = self.base_post_size * (1.0 + 1.5 * urgency + 1.0 * inv_ratio)
        raw = min(raw, self.max_post_size)

        qty = self._round_lot(raw)
        return max(self.min_clip, qty)

    def compute_unwind_quotes(
        self,
        best_bid,
        best_ask,
        inventory,
        bid_sizes=None,
        ask_sizes=None,
        urgency=0.0
    ):
        """
        Returns one-sided passive quoting instructions.
        Long inventory -> post asks
        Short inventory -> post bids
        Flat -> no quote
        """
        if inventory == 0:
            return {
                'bid_price': None,
                'ask_price': None,
                'bid_size': 0,
                'ask_size': 0
            }

        qty = self._compute_post_size(inventory, urgency)
        qty = min(qty, abs(inventory))
        qty = self._round_lot(qty)

        if qty < 100:
            return {
                'bid_price': None,
                'ask_price': None,
                'bid_size': 0,
                'ask_size': 0
            }

        spread = round(best_ask - best_bid, 2)

        # Long inventory: sell passively, maybe slightly improve ask if urgency is high
        if inventory > 0:
            if urgency < 0.35:
                ask_price = round(best_ask, 2)
            elif urgency < 0.70:
                ask_price = round(max(best_bid + 0.01, best_ask - self.price_improve), 2)
            else:
                ask_price = round(max(best_bid + 0.01, best_bid + 0.02), 2)

            return {
                'bid_price': None,
                'ask_price': ask_price,
                'bid_size': 0,
                'ask_size': qty
            }

        # Short inventory: buy back passively
        else:
            if urgency < 0.35:
                bid_price = round(best_bid, 2)
            elif urgency < 0.70:
                bid_price = round(min(best_ask - 0.01, best_bid + self.price_improve), 2)
            else:
                bid_price = round(min(best_ask - 0.01, best_ask - 0.02), 2)

            return {
                'bid_price': bid_price,
                'ask_price': None,
                'bid_size': qty,
                'ask_size': 0
            }

In [5]:
class Hitter:
    """
    For selling long inventory using marketable limit SELL orders.
    We cross the bid, but cap how deep we are willing to go.
    """

    def __init__(self, max_clip=10000, protection_cents=0.06):
        self.max_clip = max_clip
        self.protection_cents = protection_cents

    def _round_lot(self, qty):
        return max(0, int(qty / 100) * 100)

    def hit_bid(self, best_bid, inventory, desired_qty=None):
        if inventory <= 0:
            return None

        qty = abs(inventory) if desired_qty is None else min(abs(inventory), desired_qty)
        qty = min(qty, self.max_clip)
        qty = self._round_lot(qty)

        if qty < 100:
            return None

        # marketable limit sell: limit price slightly below best bid
        limit_price = round(best_bid - self.protection_cents, 2)

        return {
            'action': 'SELL',
            'quantity': qty,
            'price': limit_price
        }


class Lifter:
    """
    For buying back short inventory using marketable limit BUY orders.
    """

    def __init__(self, max_clip=10000, protection_cents=0.06):
        self.max_clip = max_clip
        self.protection_cents = protection_cents

    def _round_lot(self, qty):
        return max(0, int(qty / 100) * 100)

    def lift_ask(self, best_ask, inventory, desired_qty=None):
        if inventory >= 0:
            return None

        qty = abs(inventory) if desired_qty is None else min(abs(inventory), desired_qty)
        qty = min(qty, self.max_clip)
        qty = self._round_lot(qty)

        if qty < 100:
            return None

        # marketable limit buy: limit price slightly above best ask
        limit_price = round(best_ask + self.protection_cents, 2)

        return {
            'action': 'BUY',
            'quantity': qty,
            'price': limit_price
        }

In [6]:
class MarketMonitor:
    def __init__(
        self,
        max_history=40,
        wide_spread_threshold=0.12,
        thin_depth_threshold=4000,
        imbalance_threshold=0.65
    ):
        self.max_history = max_history
        self.wide_spread_threshold = wide_spread_threshold
        self.thin_depth_threshold = thin_depth_threshold
        self.imbalance_threshold = imbalance_threshold

        self.recent_spreads = deque(maxlen=max_history)
        self.recent_bid_depth = deque(maxlen=max_history)
        self.recent_ask_depth = deque(maxlen=max_history)
        self.recent_mid = deque(maxlen=max_history)

    def record_book(self, best_bid, best_ask, bid_sizes=None, ask_sizes=None):
        spread = round(best_ask - best_bid, 4)
        self.recent_spreads.append(spread)

        top_bid_depth = bid_sizes[0] if bid_sizes and len(bid_sizes) > 0 else 0
        top_ask_depth = ask_sizes[0] if ask_sizes and len(ask_sizes) > 0 else 0

        self.recent_bid_depth.append(top_bid_depth)
        self.recent_ask_depth.append(top_ask_depth)
        self.recent_mid.append((best_bid + best_ask) / 2)

    def detect_wide_spread(self):
        if not self.recent_spreads:
            return False
        return self.recent_spreads[-1] >= self.wide_spread_threshold

    def detect_thin_liquidity(self, side=None):
        """
        side='SELL' means we care about bid-side depth
        side='BUY' means we care about ask-side depth
        """
        if side == 'SELL':
            if not self.recent_bid_depth:
                return False
            return self.recent_bid_depth[-1] < self.thin_depth_threshold

        if side == 'BUY':
            if not self.recent_ask_depth:
                return False
            return self.recent_ask_depth[-1] < self.thin_depth_threshold

        if not self.recent_bid_depth or not self.recent_ask_depth:
            return False

        return (
            self.recent_bid_depth[-1] < self.thin_depth_threshold
            or self.recent_ask_depth[-1] < self.thin_depth_threshold
        )

    def book_imbalance(self):
        if not self.recent_bid_depth or not self.recent_ask_depth:
            return 0.0

        bid_depth = self.recent_bid_depth[-1]
        ask_depth = self.recent_ask_depth[-1]
        total = bid_depth + ask_depth

        if total <= 0:
            return 0.0

        # positive means stronger bid side, negative means stronger ask side
        return (bid_depth - ask_depth) / total

    def favorable_for_selling(self):
        return self.book_imbalance() >= self.imbalance_threshold

    def favorable_for_buying(self):
        return self.book_imbalance() <= -self.imbalance_threshold

    def mid_trend(self, lookback=5):
        if len(self.recent_mid) < lookback:
            return 0

        mids = list(self.recent_mid)[-lookback:]
        if all(mids[i] <= mids[i + 1] for i in range(len(mids) - 1)):
            return 1
        if all(mids[i] >= mids[i + 1] for i in range(len(mids) - 1)):
            return -1
        return 0

    def reset(self):
        self.recent_spreads.clear()
        self.recent_bid_depth.clear()
        self.recent_ask_depth.clear()
        self.recent_mid.clear()

In [7]:
class TraderUI:
    def __init__(self):
        if tk._default_root is not None:
            try:
                tk._default_root.destroy()
            except Exception:
                pass

        self.root = tk.Tk()
        self.root.title('LT2 Dashboard')
        self.root.geometry('620x470')
        self.root.configure(bg='#111111')
        self.root.resizable(False, False)

        self.gateway = None
        self.stop_requested = False

        title = tk.Label(
            self.root,
            text='LT2 UNWIND DASHBOARD',
            font=('Helvetica', 17, 'bold'),
            fg='white',
            bg='#111111'
        )
        title.pack(pady=10)

        self.position_label = self.make_label('Position: 0')
        self.pnl_label = self.make_label('PnL: 0.00')
        self.phase_label = self.make_label('Phase: PRE-SECOND-BLOCK')
        self.mode_label = self.make_label('Mode: Passive Unwind')
        self.market_label = self.make_label('Market: - / -')
        self.depth_label = self.make_label('Top Depth: - / -')
        self.order_label = self.make_label('Working Order: -')
        self.urgency_label = self.make_label('Urgency: 0.00')
        self.imbalance_label = self.make_label('Book Imbalance: 0.00')

        self.alert_box = tk.Label(
            self.root,
            text='STATUS: NORMAL',
            font=('Helvetica', 14, 'bold'),
            fg='white',
            bg='dark green',
            width=34,
            height=2
        )
        self.alert_box.pack(pady=16)

        button_frame = tk.Frame(self.root, bg='#111111')
        button_frame.pack(pady=8)

        self.kill_button = tk.Button(
            button_frame,
            text='KILL ALL ORDERS',
            font=('Helvetica', 12, 'bold'),
            bg='firebrick',
            fg='white',
            width=18,
            command=self.kill_all_orders
        )
        self.kill_button.grid(row=0, column=0, padx=8)

        self.stop_button = tk.Button(
            button_frame,
            text='STOP LOOP',
            font=('Helvetica', 12, 'bold'),
            bg='gray25',
            fg='white',
            width=12,
            command=self.request_stop
        )
        self.stop_button.grid(row=0, column=1, padx=8)

    def set_gateway(self, gateway):
        self.gateway = gateway

    def request_stop(self):
        self.stop_requested = True
        self.alert_box.config(text='STOP REQUESTED', bg='gray25', fg='white')

    def kill_all_orders(self):
        if self.gateway is None:
            self.alert_box.config(text='NO GATEWAY SET', bg='orange', fg='black')
            return
        try:
            self.gateway.cancel_all_orders()
            self.alert_box.config(text='ALL ORDERS CANCELLED', bg='purple', fg='white')
        except Exception as e:
            self.alert_box.config(text=f'KILL FAILED: {str(e)[:28]}', bg='red', fg='white')

    def make_label(self, text):
        label = tk.Label(
            self.root,
            text=text,
            font=('Helvetica', 12),
            fg='white',
            bg='#111111',
            anchor='w'
        )
        label.pack(fill='x', padx=20, pady=3)
        return label

    def update(
        self,
        inventory,
        pnl,
        phase,
        mode,
        best_bid,
        best_ask,
        bid_depth,
        ask_depth,
        order,
        urgency,
        imbalance,
        alert_text='STATUS: NORMAL',
        alert_bg='dark green'
    ):
        self.position_label.config(text=f'Position: {inventory}')
        self.pnl_label.config(text=f'PnL: {pnl:.2f}')
        self.phase_label.config(text=f'Phase: {phase}')
        self.mode_label.config(text=f'Mode: {mode}')
        self.market_label.config(text=f'Market: {best_bid:.2f} / {best_ask:.2f}')
        self.depth_label.config(text=f'Top Depth: {bid_depth} / {ask_depth}')
        self.urgency_label.config(text=f'Urgency: {urgency:.2f}')
        self.imbalance_label.config(text=f'Book Imbalance: {imbalance:.2f}')

        if order is None:
            self.order_label.config(text='Working Order: -')
        else:
            self.order_label.config(
                text=f"Working Order: {order.get('action', '-')}"
                     f" {order.get('quantity', '-')}"
                     f" @ {order.get('price', '-')}"
            )

        self.alert_box.config(text=alert_text, bg=alert_bg, fg='white')

        self.root.update_idletasks()
        self.root.update()

In [13]:
class TWAPScheduler:
    """
    Simple LT2 TWAP scheduler.

    Idea:
    - Start with an initial inventory size.
    - Choose a start tick and end tick for liquidation.
    - At any elapsed time, compute how much inventory should remain
      if we were unwinding linearly through time.
    """

    def __init__(
        self,
        start_tick=0,
        end_tick=230,
        catchup_sensitivity=1.25,
        min_progress_buffer=1000
    ):
        self.start_tick = start_tick
        self.end_tick = end_tick
        self.catchup_sensitivity = catchup_sensitivity
        self.min_progress_buffer = min_progress_buffer

        self.initial_inventory = None
        self.side = None  # 'LONG' or 'SHORT'

    def reset(self):
        self.initial_inventory = None
        self.side = None

    def initialize(self, inventory):
        if inventory == 0:
            return
        if self.initial_inventory is None:
            self.initial_inventory = abs(inventory)
            self.side = 'LONG' if inventory > 0 else 'SHORT'

    def progress_ratio(self, elapsed):
        if elapsed <= self.start_tick:
            return 0.0
        if elapsed >= self.end_tick:
            return 1.0
        return (elapsed - self.start_tick) / max(1, self.end_tick - self.start_tick)

    def target_remaining(self, elapsed):
        """
        How much inventory should still be left at this time
        under a linear TWAP schedule.
        """
        if self.initial_inventory is None:
            return 0

        progress = self.progress_ratio(elapsed)
        remaining = self.initial_inventory * (1.0 - progress)
        return max(0, int(round(remaining / 100.0)) * 100)

    def schedule_gap(self, inventory, elapsed):
        """
        Positive gap means you are behind schedule and need to trade faster.
        """
        if self.initial_inventory is None:
            self.initialize(inventory)

        actual_remaining = abs(inventory)
        target_remaining = self.target_remaining(elapsed)
        gap = actual_remaining - target_remaining
        return gap

    def behind_schedule(self, inventory, elapsed):
        gap = self.schedule_gap(inventory, elapsed)
        return gap > self.min_progress_buffer

    def urgency_boost(self, inventory, elapsed):
        """
        Returns a number in [0, 1] indicating how much extra urgency
        to add because you are behind schedule.
        """
        if self.initial_inventory is None:
            self.initialize(inventory)

        if self.initial_inventory is None or self.initial_inventory == 0:
            return 0.0

        gap = max(0, self.schedule_gap(inventory, elapsed))
        boost = (gap / self.initial_inventory) * self.catchup_sensitivity
        return min(1.0, max(0.0, boost))

    def target_clip(self, inventory, elapsed, base_clip=2000, max_clip=12000):
        """
        Suggests a trade size based on how far behind schedule we are.
        """
        if self.initial_inventory is None:
            self.initialize(inventory)

        gap = max(0, self.schedule_gap(inventory, elapsed))
        clip = base_clip + 0.5 * gap
        clip = min(max_clip, clip)
        clip = min(abs(inventory), clip)

        clip = int(clip / 100) * 100
        return max(0, clip)

: 

In [8]:
class Trader:
    def __init__(
        self,
        quoter,
        hitter,
        lifter,
        monitor,
        position_limit=50000,
        second_block_time=120,      # around 2-minute mark
        hard_flatten_time=230,
        final_flatten_time=285,
        max_passive_outstanding=12000,
        emergency_buffer=2000,
        twap = None
    ):
        self.quoter = quoter
        self.hitter = hitter
        self.lifter = lifter
        self.monitor = monitor

        self.position_limit = position_limit
        self.second_block_time = second_block_time
        self.hard_flatten_time = hard_flatten_time
        self.final_flatten_time = final_flatten_time
        self.max_passive_outstanding = max_passive_outstanding
        self.emergency_buffer = emergency_buffer

        self.twap = twap

        self.initial_inventory_seen = False
        self.second_block_seen = False
        self.last_abs_inventory = None

    def detect_phase(self, elapsed):
        if elapsed is None:
            return 'PRE-SECOND-BLOCK'
        if elapsed < self.second_block_time:
            return 'PRE-SECOND-BLOCK'
        if elapsed < self.hard_flatten_time:
            return 'POST-SECOND-BLOCK'
        if elapsed < self.final_flatten_time:
            return 'HARD-FLATTEN'
        return 'FINAL-FLATTEN'

    def update_block_state(self, inventory, elapsed):
        abs_inv = abs(inventory)

        if not self.initial_inventory_seen and abs_inv > 0:
            self.initial_inventory_seen = True

        if self.last_abs_inventory is not None:
            jump = abs_inv - self.last_abs_inventory
            if elapsed is not None and elapsed >= self.second_block_time and jump >= 30000:
                self.second_block_seen = True

        self.last_abs_inventory = abs_inv

    def inventory_side(self, inventory):
        if inventory > 0:
            return 'LONG'
        elif inventory < 0:
            return 'SHORT'
        return 'FLAT'

    def compute_urgency(self, inventory, elapsed):
        abs_inv = abs(inventory)
        inv_ratio = min(1.0, abs_inv / self.position_limit)

        if elapsed is None:
            base_urgency = inv_ratio
        elif elapsed < self.second_block_time:
            time_ratio = elapsed / max(1, self.second_block_time)
            base_urgency = 0.45 * inv_ratio + 0.55 * time_ratio
        elif elapsed < self.hard_flatten_time:
            time_ratio = (elapsed - self.second_block_time) / max(1, self.hard_flatten_time - self.second_block_time)
            base_urgency = 0.55 * inv_ratio + 0.45 * time_ratio
        elif elapsed < self.final_flatten_time:
            time_ratio = (elapsed - self.hard_flatten_time) / max(1, self.final_flatten_time - self.hard_flatten_time)
            base_urgency = 0.70 * inv_ratio + 0.30 * time_ratio
        else:
            base_urgency = 1.0

        twap_boost = 0.0
        if self.twap is not None and elapsed is not None:
            twap_boost = self.twap.urgency_boost(inventory, elapsed)

        urgency = base_urgency + 0.75 * twap_boost
        return min(1.0, max(0.0, urgency))

    def should_force_aggression(self, inventory, elapsed):
        abs_inv = abs(inventory)

        if abs_inv == 0:
            return False

        if elapsed is None:
            return abs_inv >= 40000

        if elapsed >= self.final_flatten_time:
            return True

        if elapsed >= self.hard_flatten_time and abs_inv > 5000:
            return True

        if elapsed >= self.second_block_time and abs_inv > 25000:
            return True

        # TWAP catch-up logic
        if self.twap is not None and self.twap.behind_schedule(inventory, elapsed):
            return True

        return False

    def choose_aggressive_size(self, inventory, elapsed, thin_liquidity):
        abs_inv = abs(inventory)

        if elapsed is None:
            base = 4000
        elif elapsed < self.second_block_time:
            base = 5000
        elif elapsed < self.hard_flatten_time:
            base = 7000
        elif elapsed < self.final_flatten_time:
            base = 10000
        else:
            base = 15000

        # TWAP catch-up clip
        if self.twap is not None and elapsed is not None:
            base = max(base, self.twap.target_clip(inventory, elapsed, base_clip=base, max_clip=15000))

        if thin_liquidity:
            base = int(base * 0.6)

        base = min(base, abs_inv)
        return max(500, int(base / 100) * 100)

    def decide_action(
        self,
        best_bid,
        best_ask,
        inventory,
        elapsed=None,
        bid_sizes=None,
        ask_sizes=None
    ):
        self.update_block_state(inventory, elapsed)
        self.monitor.record_book(best_bid, best_ask, bid_sizes=bid_sizes, ask_sizes=ask_sizes)

        phase = self.detect_phase(elapsed)
        urgency = self.compute_urgency(inventory, elapsed)

        twap_target = None
        twap_gap = None

        if self.twap is not None and elapsed is not None:
            self.twap.initialize(inventory)
            twap_target = self.twap.target_remaining(elapsed)
            twap_gap = self.twap.schedule_gap(inventory, elapsed)
        
        imbalance = self.monitor.book_imbalance()

        if inventory == 0:
            return {
                'phase': phase,
                'mode': 'Flat',
                'urgency': urgency,
                'imbalance': imbalance,
                'twap_target': twap_target,
                'twap_gap': twap_gap,
                'order': None,
                'alert_text': 'FLAT',
                'alert_bg': 'dark green'
            }

        unwind_action = 'SELL' if inventory > 0 else 'BUY'
        thin_liquidity = self.monitor.detect_thin_liquidity(side=unwind_action)
        wide_spread = self.monitor.detect_wide_spread()

        # Hard safety: if somehow close to limit and still carrying risk, get more aggressive.
        if abs(inventory) >= self.position_limit - self.emergency_buffer:
            qty = self.choose_aggressive_size(inventory, elapsed, thin_liquidity)
            if inventory > 0:
                order = self.hitter.hit_bid(best_bid, inventory, desired_qty=qty)
                return {
                    'phase': phase,
                    'mode': 'Protected Aggressive Sell',
                    'urgency': urgency,
                    'imbalance': imbalance,
                    'twap_target': twap_target,
                    'twap_gap': twap_gap,
                    'order': order,
                    'alert_text': 'NEAR LIMIT - SELL DOWN',
                    'alert_bg': 'red'
                }
            else:
                order = self.lifter.lift_ask(best_ask, inventory, desired_qty=qty)
                return {
                    'phase': phase,
                    'mode': 'Protected Aggressive Buy',
                    'urgency': urgency,
                    'imbalance': imbalance,
                    'twap_target': twap_target,
                    'twap_gap': twap_gap,
                    'order': order,
                    'alert_text': 'NEAR LIMIT - BUY BACK',
                    'alert_bg': 'red'
                }

        # Time-driven aggression
        if self.should_force_aggression(inventory, elapsed):
            qty = self.choose_aggressive_size(inventory, elapsed, thin_liquidity)

            if inventory > 0:
                order = self.hitter.hit_bid(best_bid, inventory, desired_qty=qty)
                return {
                    'phase': phase,
                    'mode': 'Protected Aggressive Sell',
                    'urgency': urgency,
                    'imbalance': imbalance,
                    'twap_target': twap_target,
                    'twap_gap': twap_gap,
                    'order': order,
                    'alert_text': 'FORCED SELL-DOWN',
                    'alert_bg': 'orange'
                }
            else:
                order = self.lifter.lift_ask(best_ask, inventory, desired_qty=qty)
                return {
                    'phase': phase,
                    'mode': 'Protected Aggressive Buy',
                    'urgency': urgency,
                    'imbalance': imbalance,
                    'twap_target': twap_target,
                    'twap_gap': twap_gap,
                    'order': order,
                    'alert_text': 'FORCED BUY-BACK',
                    'alert_bg': 'orange'
                }

        # Opportunistically be more aggressive when the book favors your unwind direction
        if inventory > 0 and self.monitor.favorable_for_selling() and not thin_liquidity:
            qty = self.choose_aggressive_size(inventory, elapsed, thin_liquidity=False)
            qty = min(qty, 5000)
            order = self.hitter.hit_bid(best_bid, inventory, desired_qty=qty)
            return {
                'phase': phase,
                'mode': 'Opportunistic Aggressive Sell',
                'urgency': urgency,
                'imbalance': imbalance,
                'twap_target': twap_target,
                'twap_gap': twap_gap,
                'order': order,
                'alert_text': 'STRONG BID SIDE',
                'alert_bg': 'dark blue'
            }

        if inventory < 0 and self.monitor.favorable_for_buying() and not thin_liquidity:
            qty = self.choose_aggressive_size(inventory, elapsed, thin_liquidity=False)
            qty = min(qty, 5000)
            order = self.lifter.lift_ask(best_ask, inventory, desired_qty=qty)
            return {
                'phase': phase,
                'mode': 'Opportunistic Aggressive Buy',
                'urgency': urgency,
                'imbalance': imbalance,
                'twap_target': twap_target,
                'twap_gap': twap_gap,
                'order': order,
                'alert_text': 'STRONG ASK SIDE',
                'alert_bg': 'dark blue'
            }

        # Otherwise: passive one-sided unwind
        quotes = self.quoter.compute_unwind_quotes(
            best_bid=best_bid,
            best_ask=best_ask,
            inventory=inventory,
            bid_sizes=bid_sizes,
            ask_sizes=ask_sizes,
            urgency=urgency
        )

        # Convert quote object into submit-order object
        if inventory > 0 and quotes['ask_size'] > 0:
            order = {
                'action': 'SELL',
                'quantity': quotes['ask_size'],
                'price': quotes['ask_price']
            }
        elif inventory < 0 and quotes['bid_size'] > 0:
            order = {
                'action': 'BUY',
                'quantity': quotes['bid_size'],
                'price': quotes['bid_price']
            }
        else:
            order = None

        alert_text = 'PASSIVE UNWIND'
        alert_bg = 'dark green'

        if wide_spread:
            alert_text = 'WIDE SPREAD'
            alert_bg = 'purple'
        if thin_liquidity:
            alert_text = 'THIN BOOK'
            alert_bg = 'brown'

        return {
            'phase': phase,
            'mode': 'Passive Unwind',
            'urgency': urgency,
            'imbalance': imbalance,
            'twap_target': twap_target,
            'twap_gap': twap_gap,
            'order': order,
            'alert_text': alert_text,
            'alert_bg': alert_bg
        }

    def step(
        self,
        best_bid,
        best_ask,
        inventory,
        elapsed=None,
        bid_sizes=None,
        ask_sizes=None
    ):
        return self.decide_action(
            best_bid=best_bid,
            best_ask=best_ask,
            inventory=inventory,
            elapsed=elapsed,
            bid_sizes=bid_sizes,
            ask_sizes=ask_sizes
        )

In [9]:
# Help functions
def get_best_bid_ask(book):
    bids = book.get('bids', [])
    asks = book.get('asks', [])

    best_bid = float(bids[0]['price']) if bids else None
    best_ask = float(asks[0]['price']) if asks else None

    return best_bid, best_ask


def estimate_pnl_from_security(sec, mid_price):
    if sec is None:
        return 0.0

    if 'realized' in sec and 'unrealized' in sec:
        try:
            return float(sec['realized']) + float(sec['unrealized'])
        except Exception:
            pass

    position = float(sec.get('position', 0))
    return position * mid_price


# Signal handler for graceful shutdown
shutdown = False


def signal_handler(signum, frame):
    global shutdown
    signal.signal(signal.SIGINT, signal.SIG_DFL)
    shutdown = True


In [10]:
def run_live_trader_logged(gateway, trader, ui, ticker='GTA', poll_delay=0.4):
    ui.set_gateway(gateway)

    loop_log = []
    order_log = []
    fill_log = []

    prev_inventory = None
    prev_mid = None
    prev_mode = None

    while True:
        try:
            if ui.stop_requested:
                print('Stop requested from UI.')
                try:
                    gateway.cancel_all_orders()
                    log_order(
                        order_log, None, ticker,
                        'cancel_all', None,
                        None, None, None, None, None
                    )
                except Exception:
                    pass
                break

            case_data = gateway.get_case()
            tick = case_data.get('tick')
            status = case_data.get('status')

            if status != 'ACTIVE':
                print('Case not active.')
                break

            # --- book ---
            book = gateway.get_book(ticker)
            best_bid, best_ask = get_best_bid_ask(book)

            if best_bid is None or best_ask is None:
                time.sleep(poll_delay)
                continue

            bid_sizes = [level['quantity'] for level in book.get('bids', [])]
            ask_sizes = [level['quantity'] for level in book.get('asks', [])]

            # --- security / inventory / pnl ---
            sec = gateway.get_security(ticker)
            inventory = int(float(sec.get('position', 0))) if sec else 0
            mid_price = (best_bid + best_ask) / 2
            pnl = estimate_pnl_from_security(sec, mid_price)

            # --- use tick as elapsed time ---
            elapsed = tick

            # --- trader decision ---
            decision = trader.step(
                best_bid=best_bid,
                best_ask=best_ask,
                inventory=inventory,
                elapsed=elapsed,
                bid_sizes=bid_sizes,
                ask_sizes=ask_sizes
            )

            mode = decision['mode']
            order = decision['order']

            # --- infer fills from inventory changes ---
            if prev_inventory is not None and prev_mid is not None and prev_mode is not None:
                infer_and_log_fills(
                    fill_log=fill_log,
                    tick=tick,
                    prev_inventory=prev_inventory,
                    inventory=inventory,
                    best_bid=best_bid,
                    best_ask=best_ask,
                    prev_mid=prev_mid,
                    mode=prev_mode
                )

            # --- always cancel stale orders first in LT2 ---
            try:
                gateway.cancel_all_orders()
                log_order(
                    order_log, tick, ticker,
                    'cancel_all', mode,
                    None, None, None, None, inventory
                )
            except Exception as e:
                print('Cancel failed:', e)

            # --- emergency flatten if over hard limit ---
            if abs(inventory) > trader.position_limit:
                flatten_qty = min(abs(inventory), 5000)
                flatten_qty = max(100, int(flatten_qty / 100) * 100)

                if inventory > 0:
                    px = round(best_bid - 0.05, 2)
                    gateway.submit_order(
                        ticker=ticker,
                        order_type='LIMIT',
                        quantity=flatten_qty,
                        action='SELL',
                        price=px
                    )
                    log_order(
                        order_log, tick, ticker,
                        'submit', 'Emergency Flatten',
                        'LIMIT', 'SELL', flatten_qty, px, inventory
                    )
                else:
                    px = round(best_ask + 0.05, 2)
                    gateway.submit_order(
                        ticker=ticker,
                        order_type='LIMIT',
                        quantity=flatten_qty,
                        action='BUY',
                        price=px
                    )
                    log_order(
                        order_log, tick, ticker,
                        'submit', 'Emergency Flatten',
                        'LIMIT', 'BUY', flatten_qty, px, inventory
                    )

            # --- normal LT2 execution ---
            elif order is not None:
                qty = max(0, int(order['quantity'] / 100) * 100)

                if order['action'] == 'BUY':
                    max_qty = max(0, -inventory) if inventory < 0 else 0
                    qty = min(qty, max_qty)

                elif order['action'] == 'SELL':
                    max_qty = max(0, inventory) if inventory > 0 else 0
                    qty = min(qty, max_qty)

                if qty >= 100:
                    gateway.submit_order(
                        ticker=ticker,
                        order_type='LIMIT',
                        quantity=qty,
                        action=order['action'],
                        price=order['price']
                    )

                    log_order(
                        order_log, tick, ticker,
                        'submit', mode,
                        'LIMIT', order['action'], qty,
                        order['price'], inventory
                    )

            # --- logging ---
            log_loop(loop_log, tick, best_bid, best_ask, inventory, pnl, decision)

            # --- UI update ---
            ui.update(
                inventory=inventory,
                pnl=pnl,
                phase=decision['phase'],
                mode=decision['mode'],
                best_bid=best_bid,
                best_ask=best_ask,
                bid_depth=bid_sizes[0] if bid_sizes else 0,
                ask_depth=ask_sizes[0] if ask_sizes else 0,
                order=order,
                urgency=decision['urgency'],
                imbalance=decision['imbalance'],
                alert_text=decision.get('alert_text', 'STATUS: NORMAL'),
                alert_bg=decision.get('alert_bg', 'dark green')
            )

            print(
                f"tick={tick} | elapsed={elapsed} | inv={inventory} | "
                f"bid={best_bid:.2f} | ask={best_ask:.2f} | "
                f"mode={decision['mode']} | pnl={pnl:.2f}"
            )

            prev_inventory = inventory
            prev_mid = mid_price
            prev_mode = mode

            time.sleep(poll_delay)

        except KeyboardInterrupt:
            print('Stopping trader...')
            try:
                gateway.cancel_all_orders()
                log_order(
                    order_log,
                    tick if 'tick' in locals() else None,
                    ticker,
                    'cancel_all',
                    None,
                    None, None, None, None, None
                )
            except Exception:
                pass
            break

        except Exception as e:
            print('Error in live trader:', e)
            time.sleep(1)

    loop_df = pd.DataFrame(loop_log)
    order_df = pd.DataFrame(order_log)
    fill_df = pd.DataFrame(fill_log)

    return loop_df, order_df, fill_df

In [11]:
gateway = RITGateway('user4', 'user4')

quoter = Quoter(
    base_post_size=2000,
    max_post_size=8000,
    inventory_limit=50000,
    min_clip=500,
    price_improve=0.01
)

hitter = Hitter(
    max_clip=12000,
    protection_cents=0.06
)

lifter = Lifter(
    max_clip=12000,
    protection_cents=0.06
)

monitor = MarketMonitor(
    max_history=40,
    wide_spread_threshold=0.12,
    thin_depth_threshold=4000,
    imbalance_threshold=0.65
)

twap = TWAPScheduler(
    start_tick=0,
    end_tick=230,
    catchup_sensitivity=1.25,
    min_progress_buffer=1000
)

trader = Trader(
    quoter=quoter,
    hitter=hitter,
    lifter=lifter,
    monitor=monitor,
    position_limit=50000,
    second_block_time=120,
    hard_flatten_time=230,
    final_flatten_time=285,
    max_passive_outstanding=12000,
    emergency_buffer=2000,
    twap=twap
)

In [12]:
ui = TraderUI()

loop_df, order_df, fill_df = run_live_trader_logged(
    gateway=gateway,
    trader=trader,
    ui=ui,
    ticker='GTA',
    poll_delay=0.4
)

Case not active.


In [ ]:
def analyze_run(loop_df, order_df, fill_df):
    print('===== SUMMARY =====')
    if not loop_df.empty:
        final_row = loop_df.iloc[-1]
        print(f"Final tick: {final_row['tick']}")
        print(f"Final PnL: {final_row['pnl']:.2f}")
        print(f"Final inventory: {final_row['inventory']}")
        print(f"Max abs inventory: {loop_df['inventory'].abs().max()}")
        print(f"Average abs inventory: {loop_df['inventory'].abs().mean():.2f}")
        print(f"Time with |inventory| > 3000: {(loop_df['inventory'].abs() > 3000).sum()} ticks")
        print()

        mode_counts = loop_df['mode'].value_counts(dropna=False)
        print('Mode counts:')
        print(mode_counts)
        print()

        pnl_by_mode = loop_df.groupby('mode')['pnl'].agg(['count', 'mean', 'max', 'min'])
        print('PnL by mode:')
        print(pnl_by_mode)
        print()

    if not order_df.empty:
        print('Order event counts:')
        print(order_df.groupby(['event_type', 'order_type', 'action']).size())
        print()

    if not fill_df.empty:
        print('Estimated fill summary:')
        print(f"Total estimated filled shares: {fill_df['qty'].sum()}")
        print(f"Average estimated edge/share: {fill_df['edge_per_share'].mean():.4f}")
        print(f"Total estimated edge dollars: {fill_df['estimated_edge_dollars'].sum():.2f}")
        print()

        print('Estimated fills by mode:')
        print(fill_df.groupby('mode').agg(
            fills=('qty', 'sum'),
            avg_edge_per_share=('edge_per_share', 'mean'),
            total_edge_dollars=('estimated_edge_dollars', 'sum')
        ))
        print()

    # plots
    if not loop_df.empty:
        plt.figure(figsize=(10, 4))
        plt.plot(loop_df['tick'], loop_df['pnl'])
        plt.title('PnL over Tick')
        plt.xlabel('Tick')
        plt.ylabel('PnL')
        plt.show()

        plt.figure(figsize=(10, 4))
        plt.plot(loop_df['tick'], loop_df['inventory'])
        plt.title('Inventory over Tick')
        plt.xlabel('Tick')
        plt.ylabel('Inventory')
        plt.show()

        plt.figure(figsize=(10, 4))
        plt.plot(loop_df['tick'], loop_df['mid'], label='Mid')
        if loop_df['quoted_bid'].notna().any():
            plt.plot(loop_df['tick'], loop_df['quoted_bid'], label='Quoted Bid', alpha=0.8)
        if loop_df['quoted_ask'].notna().any():
            plt.plot(loop_df['tick'], loop_df['quoted_ask'], label='Quoted Ask', alpha=0.8)
        plt.title('Mid and Quotes over Tick')
        plt.xlabel('Tick')
        plt.ylabel('Price')
        plt.legend()
        plt.show()

    if not fill_df.empty:
        plt.figure(figsize=(8, 4))
        plt.hist(fill_df['edge_per_share'].dropna(), bins=30)
        plt.title('Estimated Edge per Share Histogram')
        plt.xlabel('Edge per Share')
        plt.ylabel('Count')
        plt.show()

In [ ]:
def save_run_logs(loop_df, order_df, fill_df, prefix='lt1_run'):
    loop_df.to_csv(f'{prefix}_loop.csv', index=False)
    order_df.to_csv(f'{prefix}_orders.csv', index=False)
    fill_df.to_csv(f'{prefix}_fills.csv', index=False)
    print(f"Saved: {prefix}_loop.csv, {prefix}_orders.csv, {prefix}_fills.csv")

In [ ]:
analyze_run(loop_df, order_df, fill_df)
save_run_logs(loop_df, order_df, fill_df, prefix='har_test_1')